In [1]:
from pathlib import Path
import dask.array as da
import pandas as pd
import anndata as ad
import napari
import numpy as np
from tqdm.auto import tqdm
import json
import seaborn as sns
import matplotlib.pyplot as plt
expanded_piyg = ['#1a9641', '#a6d96a', '#978897', '#d1d1ca', '#f1b6da', '#d02c91']


def timed_compute(volume):
    """
    Compute a lazy Dask array frame-by-frame with progress reporting.

    This function iterates over the leading axis of a Dask array (e.g. time),
    calls `.compute()` on each slice, and stacks the results into a single
    NumPy array. A tqdm progress bar is displayed to indicate progress.

    Parameters
    ----------
    volume : dask.array.Array
        A Dask array with at least one dimension (e.g. shape (T, ...)).
        The function will iterate over the first axis (axis=0).

    Returns
    -------
    numpy.ndarray
        A NumPy array with the same shape as `volume`, but fully realized
        in memory. The dtype is preserved from the Dask array.

    Notes
    -----
    - Each frame is computed independently, which can be helpful for
      monitoring performance and memory use on large arrays.
    - The returned array may be very large if `volume` is large.
      Ensure sufficient memory is available.
    """
    return np.stack([frame.compute() for frame in tqdm(volume)], axis=0)


In [3]:
root_dir = Path('/mnt/OPERA3/Nathan/data/macrohet/Z_stack_tests/zarr')
# root_dir = Path('/Volumes/OPERA3/Nathan/Z_stack_tests/zarr')
rc_stem = "(3,3)"
store = root_dir / f"{rc_stem}.zarr"

# --- load images and segmentation ---
images = da.from_zarr(str(store / "images" / "0"))   # (T,C,Z,Y,X)
masks  = da.from_zarr(str(store / "labels" / "masks"))  # (T,Z,Y,X)
tracked_masks = da.from_zarr(str(store / "labels" / "masks_tracked"))  # (T,Z,Y,X)
# --- load tracks (CSV preferred, fallback to AnnData) ---
# tracks_csv = root_dir / f"{rc_stem}_tracks.csv"
# if tracks_csv.exists():
#     tracks = pd.read_csv(tracks_csv)
# else:
adata = ad.read_zarr(str(store / "tables" / "quantified_tracks"))
tracks = adata.obs.reset_index(drop=True)

print("images:", images.shape)
print("masks:", masks.shape)
print("tracked masks:", tracked_masks.shape)
print("tracks:", tracks.shape)


images: (97, 2, 25, 7992, 7992)
masks: (97, 25, 7992, 7992)
tracked masks: (97, 7992, 7992)
tracks: (1859520, 10)


In [50]:
images

dask.array<from-zarr, shape=(97, 2, 25, 7992, 7992), dtype=uint16, chunksize=(1, 1, 16, 512, 512), chunktype=numpy.ndarray>

In [5]:
tracked_masks

dask.array<from-zarr, shape=(97, 7992, 7992), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>

In [16]:
tracks = tracks[tracks['z'] == 12]
tracks

,t,ID,z,y,x,cell_area_px,mtb_area_px,segment_ID,row,col
12,0,1,12,866,6511,13475,0,817,3,3
29,0,6,12,525,4880,14200,0,755,3,3
47,0,9,12,7490,1929,11750,0,64,3,3
65,0,10,12,4221,542,21675,0,1009,3,3
83,0,11,12,4548,823,14975,18,442,3,3
...,...,...,...,...,...,...,...,...,...,...
1859434,96,8451,12,6487,2633,16700,0,361,3,3
1859449,96,8492,12,6519,109,11050,0,1758,3,3
1859470,96,8531,12,5452,1006,12250,69,646,3,3
1859487,96,8562,12,7760,1892,17300,215,1438,3,3


In [17]:
napari_tracks = tracks[['ID', 't', 'z', 'y', 'x']].to_numpy(dtype=np.int64)
napari_tracks.shape

(108600, 5)

In [9]:
viewer = napari.Viewer(title = 'inspecting tracks and quantifications')
viewer.add_image(images, channel_axis=1)
viewer.add_labels(masks)
viewer.add_tracks(napari_tracks)

<Tracks layer 'napari_tracks' at 0x7534686b68c0>

In [12]:
napari_tracks[:, 1].shape

(1859520,)

In [19]:
# features = {
#     'time': tracks['t'].values,
#     'mtb_area_px': tracks['mtb_area_px'].values
# }

features = {
    col: tracks[col].values
    for col in tracks.columns
}

In [20]:
viewer.layers['napari_tracks'].features = features

In [21]:
scale = [1.0, 0.14949, 0.14949]

In [22]:
for layer in viewer.layers:
    layer.scale = scale

In [14]:
import napari_animation

In [25]:
def update_slider(event):
    # Compute time in hours
    time = viewer.dims.current_step[0] / 2
    # Update the text overlay (bottom left)
    viewer.text_overlay.text = f"{time:1.2f} hrs"

# Configure text overlay
viewer.text_overlay.visible = True
viewer.text_overlay.color = "white"
viewer.text_overlay.font_size = 24
viewer.text_overlay.position = "bottom_left"

# Add scale bar (bottom right)
viewer.scale_bar.visible = True
viewer.scale_bar.unit = "µm"
viewer.scale_bar.ticks = False
viewer.scale_bar.colored = False
viewer.scale_bar.font_size = 24
viewer.scale_bar.color = "white"
viewer.scale_bar.position = "bottom_right"

# Connect slider update
viewer.dims.events.current_step.connect(update_slider)


<function __main__.update_slider(event)>

In [26]:
viewer.camera

Camera(center=(12.0, 597.287295, 597.287295), zoom=0.7577900329923872, angles=(0.490555016764744, 9.421260731998562, -90.20087752517617), perspective=0.0, mouse_pan=True, mouse_zoom=True)

In [27]:
from napari_animation import Animation
from tqdm.auto import tqdm
import numpy as np

# --- Movie 1: straight time-through ---
viewer.dims.ndisplay = 3

anim1 = Animation(viewer)

nT = viewer.dims.nsteps[0]
fps = 30
duration_s = 20
total_frames = int(round(duration_s * fps))
steps_between = max(1, total_frames // max(1, (nT - 1)))

# fix camera
cam = viewer.camera
cam_center, cam_zoom, cam_angles, cam_persp = tuple(cam.center), cam.zoom, tuple(cam.angles), cam.perspective

step0 = list(viewer.dims.current_step)
step0[0] = 0
viewer.dims.current_step = tuple(step0)
viewer.camera.center = cam_center
viewer.camera.zoom = cam_zoom
viewer.camera.angles = cam_angles
viewer.camera.perspective = cam_persp
anim1.capture_keyframe()

for t in tqdm(range(1, nT), desc="Capturing time keyframes"):
    step = list(step0)
    step[0] = t
    viewer.dims.current_step = tuple(step)
    anim1.capture_keyframe(steps=steps_between)

anim1.animate("timelapse_time.mp4", fps=fps, canvas_only=True, quality=9)



Capturing time keyframes:   0%|          | 0/96 [00:00<?, ?it/s]

TypeError: Animation.animate() got an unexpected keyword argument 'size'

In [31]:

# --- Movie 2: oblique tilt + zoom + pan ---
anim2 = Animation(viewer)

# baseline camera
cam0 = viewer.camera
c0 = np.array(cam0.center, dtype=float)
z0 = float(cam0.zoom)
a0 = np.array(cam0.angles, dtype=float)
p0 = float(cam0.perspective)

# targets
c1 = c0 + np.array([100.0, -100.0, 0.0])
z1 = z0 * 2.0
a1 = a0 + np.array([20.0, 45.0, 0.0])
p1 = p0

step0 = list(viewer.dims.current_step)
step0[0] = 0
viewer.dims.current_step = tuple(step0)

# initial
viewer.camera.center = tuple(c0)
viewer.camera.zoom = z0
viewer.camera.angles = tuple(a0)
viewer.camera.perspective = p0
anim2.capture_keyframe()

for t in tqdm(range(1, nT), desc="Capturing oblique keyframes"):
    u = t / (nT - 1)  # 0→1
    step = list(step0)
    step[0] = t
    viewer.dims.current_step = tuple(step)

    viewer.camera.center = tuple(c0 * (1 - u) + c1 * u)
    viewer.camera.zoom = z0 * (1 - u) + z1 * u
    viewer.camera.angles = tuple(a0 * (1 - u) + a1 * u)
    viewer.camera.perspective = p0 * (1 - u) + p1 * u

    anim2.capture_keyframe(steps=steps_between)

anim2.animate("timelapse_oblique.mp4", fps=fps, canvas_only=True, quality=9)


Capturing oblique keyframes:   0%|          | 0/96 [00:00<?, ?it/s]

Rendering frames...


100%|██████████████████████████████████████████████████████████████████████████████| 577/577 [4:20:18<00:00, 27.07s/it]


In [30]:
anim1.animate("timelapse_time.mp4", fps=fps, canvas_only=True, quality=9)


Rendering frames...


100%|██████████████████████████████████████████████████████████████████████████████| 577/577 [1:57:57<00:00, 12.27s/it]


In [33]:
viewer.camera

Camera(center=(12.0, 597.287295, 597.287295), zoom=0.7577900329923877, angles=(0.0, 0.0, 89.99999999999999), perspective=0.0, mouse_pan=True, mouse_zoom=True)

In [32]:
viewer.camera

Camera(center=(27.049973349555515, 517.4284348868151, 543.4135819134057), zoom=5.8031778181499964, angles=(53.02429642782195, -56.04043051486555, -52.9144704854027), perspective=0.0, mouse_pan=True, mouse_zoom=True)

In [34]:
from napari_animation import Animation
from tqdm.auto import tqdm
import numpy as np

viewer.dims.ndisplay = 3

# --- define endpoints from your snapshots ---
start_center = np.array((12.0, 597.287295, 597.287295), dtype=float)
start_zoom   = 0.7577900329923877
start_angles = np.array((0.0, 0.0, 89.9999999999), dtype=float)
start_persp  = 0.0

end_center = np.array((27.04997334955515, 517.4284348868151, 543.4135819134057), dtype=float)
end_zoom   = 5.8031778181499964
end_angles = np.array((53.02429642782195, -56.004403051486555, -52.9144704854027), dtype=float)
end_persp  = 0.0  # unchanged

# --- pacing: ~20 s at 30 fps while visiting every time step ---
nT = viewer.dims.nsteps[0]
fps = 30
duration_s = 20
total_frames = int(round(duration_s * fps))
steps_between = max(1, total_frames // max(1, (nT - 1)))

anim = Animation(viewer)

# go to t=0 with start camera
step0 = list(viewer.dims.current_step); step0[0] = 0
viewer.dims.current_step = tuple(step0)
viewer.camera.center = tuple(start_center)
viewer.camera.zoom = float(start_zoom)
viewer.camera.angles = tuple(start_angles)
viewer.camera.perspective = float(start_persp)
anim.capture_keyframe()

# sweep time; interpolate camera to the end pose
for t in tqdm(range(1, nT), desc="Keyframing cam (start → end)"):
    u = t / (nT - 1)  # 0→1 across the timelapse

    viewer.dims.current_step = tuple([t] + step0[1:])

    viewer.camera.center = tuple(start_center * (1 - u) + end_center * u)
    viewer.camera.zoom   = float(start_zoom * (1 - u) + end_zoom * u)
    viewer.camera.angles = tuple(start_angles * (1 - u) + end_angles * u)
    viewer.camera.perspective = float(start_persp * (1 - u) + end_persp * u)

    anim.capture_keyframe(steps=steps_between)

# render mp4
anim.animate("timelapse_4D_oblique_view.mp4", fps=fps, canvas_only=True, quality=9)


Keyframing cam (start → end):   0%|          | 0/96 [00:00<?, ?it/s]

Rendering frames...


100%|██████████████████████████████████████████████████████████████████████████████| 577/577 [5:28:21<00:00, 34.15s/it]


In [35]:
from napari_animation import Animation
from tqdm.auto import tqdm
import numpy as np

viewer.dims.ndisplay = 3

anim2 = Animation(viewer)

# Timing / pacing
nT = viewer.dims.nsteps[0]
fps = 30
duration_s = 20
total_frames = int(round(duration_s * fps))
steps_between = max(1, total_frames // max(1, (nT - 1)))

# Baseline (keep current center & perspective; start angles forced to top-down)
cam0 = viewer.camera
c0 = np.array(cam0.center, dtype=float)
z0 = float(cam0.zoom)
p0 = float(cam0.perspective)

# Start at orthogonal top-down, end at 45° oblique; no pan
a_start = np.array([90.0, 0.0, 0.0])   # (elev, azim, roll) — top-down
a_end   = np.array([45.0, 45.0, 0.0])  # tilt down + add azimuth for oblique
z_end   = z0 * 4.0                     # 4× zoom

# Go to t=0, apply start camera
step0 = list(viewer.dims.current_step)
step0[0] = 0
viewer.dims.current_step = tuple(step0)

viewer.camera.center = tuple(c0)
viewer.camera.zoom = z0
viewer.camera.angles = tuple(a_start)
viewer.camera.perspective = p0
anim2.capture_keyframe()

# Sweep time; rotate + zoom smoothly
for t in tqdm(range(1, nT), desc="Capturing oblique (rotate + ×4 zoom)"):
    u = t / (nT - 1)  # 0→1 across timelapse

    step = list(step0)
    step[0] = t
    viewer.dims.current_step = tuple(step)

    viewer.camera.center = tuple(c0)  # no pan
    viewer.camera.zoom = z0 * (1 - u) + z_end * u
    viewer.camera.angles = tuple(a_start * (1 - u) + a_end * u)
    viewer.camera.perspective = p0

    anim2.capture_keyframe(steps=steps_between)

# Render MP4
anim2.animate("timelapse_obliquer.mp4", fps=fps, canvas_only=True, quality=9)


Capturing oblique (rotate + ×4 zoom):   0%|          | 0/96 [00:00<?, ?it/s]

Rendering frames...


100%|██████████████████████████████████████████████████████████████████████████████| 577/577 [5:44:01<00:00, 35.77s/it]


In [36]:
def highlight_cell(
    cell_ID,
    viewer,
    tracks,
    *,
    scale_factor: float = 1.0,
    napari_scale=None,
    size: int = 300,
    opacity: float = 1.0,
    symbol: str = "o",
    reset_position: bool = True,
):
    """
    Highlight a cell's trajectory in napari.

    Parameters
    ----------
    cell_ID : int
        Cell ID to highlight.
    viewer : napari.viewer.Viewer
        The napari viewer instance.
    tracks : list[Tracklet] | np.ndarray | pd.DataFrame
        Either:
          • a list of Tracklet objects with attrs (ID, t, y, x), or
          • an array/DataFrame with columns ordered as ID T(Z)YX.
    scale_factor : float
        Factor to multiply Y/X coordinates.
    napari_scale : list[float] | None
        Scale metadata for napari.
    size, opacity, symbol, reset_position
        Visual options.

    Returns
    -------
    napari.layers.Points
    """

    import numpy as np
    import pandas as pd

    # --- Case 1: Tracklet objects ---
    if isinstance(tracks, (list, tuple)) and tracks and hasattr(tracks[0], "ID"):
        tr = next(t for t in tracks if t.ID == cell_ID)
        points = np.column_stack([
            np.asarray(tr.t, dtype=float),
            np.asarray(tr.y, dtype=float) * scale_factor,
            np.asarray(tr.x, dtype=float) * scale_factor,
        ])
        has_z = False

    # --- Case 2: Array/DataFrame, schema = ID T(Z)YX ---
    else:
        if isinstance(tracks, pd.DataFrame):
            arr = tracks.to_numpy(dtype=float)
            cols = list(tracks.columns)
            has_z = "z" in cols
        else:
            arr = np.asarray(tracks, dtype=float)
            has_z = arr.shape[1] == 5

        # Filter rows by ID
        arr = arr[arr[:, 0] == float(cell_ID)]

        # Drop NaN rows
        arr = arr[~np.isnan(arr).any(axis=1)]
        if arr.size == 0:
            raise ValueError(f"No valid samples for cell {cell_ID}.")

        if has_z:
            t, z, y, x = arr[:, 1], arr[:, 2], arr[:, 3] * scale_factor, arr[:, 4] * scale_factor
            points = np.column_stack([t, z, y, x])
        else:
            t, y, x = arr[:, 1], arr[:, 2] * scale_factor, arr[:, 3] * scale_factor
            points = np.column_stack([t, y, x])

    # --- Add to napari ---
    layer = viewer.add_points(
        points,
        size=size,
        symbol=symbol,
        face_color="transparent",
        edge_color="white",
        edge_width=0.1,
        name=f"cell {cell_ID}",
        opacity=opacity,
        scale=napari_scale,
    )

    if reset_position:
        step = tuple(int(round(v)) for v in points[0])
        viewer.dims.current_step = step

    return layer

In [40]:
ID = 1307
highlight_cell(ID, viewer, napari_tracks, scale_factor=0.149)


/tmp/ipykernel_862848/2025097940.py:77: FutureWarning: Argument 'edge_width' is deprecated, please use 'border_width' instead. The argument 'edge_width' was deprecated in 0.5.0 and it will be removed in 0.6.0.
  layer = viewer.add_points(
/home/dayn/miniconda3/envs/godspeed/lib/python3.10/site-packages/napari/utils/migrations.py:101: FutureWarning: Argument 'edge_color' is deprecated, please use 'border_color' instead. The argument 'edge_color' was deprecated in 0.5.0 and it will be removed in 0.6.0.
  return func(*args, **kwargs)


<Points layer 'cell 1307 [1]' at 0x7532f8208640>

In [39]:
tracks[tracks['ID'] == ID]

,t,ID,z,y,x,cell_area_px,mtb_area_px,segment_ID,row,col
3905,0,1307,12,1832,2748,13675,107,1328,3,3
18723,1,1307,12,1824,2735,13050,0,1169,3,3
33037,2,1307,12,1824,2727,13475,38,792,3,3
46486,3,1307,12,1834,2734,12800,0,898,3,3
59634,4,1307,12,1836,2721,12850,0,766,3,3
72620,5,1307,12,1855,2706,13600,0,1092,3,3
85677,6,1307,12,1842,2720,12750,0,1530,3,3
98997,7,1307,12,1847,2720,13050,0,833,3,3
112652,8,1307,12,1841,2728,12825,0,846,3,3
126682,9,1307,12,1841,2734,12775,0,1514,3,3


In [41]:
# Timing / pacing
nT = viewer.dims.nsteps[0]
fps = 30
duration_s = 20
total_frames = int(round(duration_s * fps))
steps_between = max(1, total_frames // max(1, (nT - 1)))



In [43]:

animation = Animation(viewer)

In [48]:



animation.capture_keyframe()

In [49]:
anim2.animate("timelapse_obliquer_test.mp4", fps=fps, canvas_only=True, quality=5)


Rendering frames...


  0%|▏                                                                               | 1/577 [00:40<6:31:08, 40.74s/it]


KeyboardInterrupt: 

In [ ]:


# Sweep time; rotate + zoom smoothly
for t in tqdm(range(1, nT), desc="Capturing oblique (rotate + ×4 zoom)"):
    u = t / (nT - 1)  # 0→1 across timelapse

    step = list(step0)
    step[0] = t
    viewer.dims.current_step = tuple(step)

    viewer.camera.center = tuple(c0)  # no pan
    viewer.camera.zoom = z0 * (1 - u) + z_end * u
    viewer.camera.angles = tuple(a_start * (1 - u) + a_end * u)
    viewer.camera.perspective = p0

    anim2.capture_keyframe(steps=steps_between)

# Render MP4
anim2.animate("timelapse_obliquer.mp4", fps=fps, canvas_only=True, quality=9)
